# Train Control Center

In this notebook, you will learn how the control center monitors the train network:

1. Subscribe to sensor data from stations
2. Evaluate waiting passenger counts against thresholds
3. Request extra trains from the dispatcher when needed
4. Log dispatch decisions for analysis

The control center is the "brain" that decides when extra capacity is needed on the network.

## Step 1: Setup and Configuration

Load configuration and prepare the simulation environment.

In [ ]:
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher
from simulated_city.agents import (
    Station,
    SimulationState,
    SensorAgent,
    ControlCenterAgent,
    DispatcherAgent,
)
from datetime import datetime
import time

# Load configuration
config = load_config()

# Connect to MQTT
connector = MqttConnector(config.mqtt, client_id_suffix="control-center")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
    publisher = MqttPublisher(connector)
else:
    print("✗ Failed to connect to MQTT broker")

# Display dispatcher configuration
threshold = config.train_network.dispatcher.waiting_threshold
print(f"\nDispatcher Configuration:")
print(f"  Threshold: {threshold} passengers")
print(f"  ➜ Extra trains deployed when waiting > {threshold}")

## Step 2: Create Simulation State and Stations

Initialize the simulation state that will track passenger queues.

In [ ]:
# Initialize simulation state
sim_state = SimulationState(current_time=datetime.now())

# Convert config to Station objects
route = []
for station_config in config.train_network.route:
    station = Station(
        name=station_config.name,
        station_type=station_config.type,
        location_lat=station_config.location.lat,
        location_lon=station_config.location.lon,
        exit_percentage=station_config.exit_percentage,
    )
    route.append(station)

entry_stations = [s for s in route if s.station_type == "entry"]

print(f"Monitoring {len(entry_stations)} entry stations:")
for station in entry_stations:
    print(f"  - {station.name}")

## Step 3: Create Sensor Agents

Sensors monitor each station and publish waiting passenger counts. The control center subscribes to these updates.

In [ ]:
# Create sensor agents for each entry station
sensor_agents = []

for station in entry_stations:
    sensor = SensorAgent(
        station=station,
        mqtt_publisher=publisher,
        mqtt_base_topic=config.train_network.mqtt_base_topic,
    )
    sensor_agents.append(sensor)

print(f"Created {len(sensor_agents)} sensor agents")

## Step 4: Create Control Center Agent

The control center monitors sensor data and requests extra trains when thresholds are exceeded.

In [ ]:
# Create control center
control_center = ControlCenterAgent(
    dispatcher_config=config.train_network.dispatcher,
    mqtt_connector=connector,
    mqtt_publisher=publisher,
    mqtt_base_topic=config.train_network.mqtt_base_topic,
)

# Start control center (subscribes to sensor topics)
control_center.start()

print("✓ Control center started")
print(f"  Subscribed to: {config.train_network.mqtt_base_topic}/station/+/sensor/waiting_count")

## Step 5: Simulate Different Passenger Load Scenarios

Let's test the control center's threshold logic with different passenger loads.

In [ ]:
from simulated_city.agents import Passenger

# Scenario 1: Low load (below threshold)
print("Scenario 1: Low passenger load (150 passengers)\n")

test_station = entry_stations[0]
queue = sim_state.get_station_queue(test_station.name)

# Add 150 passengers
passengers = [
    Passenger(f"p-{i}", test_station.name, route[-1].name, datetime.now())
    for i in range(150)
]
queue.add_passengers(passengers)

# Evaluate threshold
should_dispatch = control_center.evaluate_threshold(test_station.name, queue.count)

print(f"  Station: {test_station.name}")
print(f"  Waiting: {queue.count} passengers")
print(f"  Threshold: {threshold}")
print(f"  Should dispatch extra train? {should_dispatch}")
print()

In [ ]:
# Scenario 2: High load (above threshold)
print("Scenario 2: High passenger load (300 passengers)\n")

# Add 150 more passengers (total 300)
more_passengers = [
    Passenger(f"p-{i+150}", test_station.name, route[-1].name, datetime.now())
    for i in range(150)
]
queue.add_passengers(more_passengers)

# Evaluate threshold
should_dispatch = control_center.evaluate_threshold(test_station.name, queue.count)

print(f"  Station: {test_station.name}")
print(f"  Waiting: {queue.count} passengers")
print(f"  Threshold: {threshold}")
print(f"  Should dispatch extra train? {should_dispatch}")

if should_dispatch:
    print(f"\n  ⚠️ THRESHOLD EXCEEDED! Requesting extra train...")
    control_center.request_extra_train(test_station.name, queue.count)
    print(f"  ✓ Dispatch request published to MQTT")

print()

## Step 6: Simulate Sensor Publishing and Control Center Response

Now let's have sensors publish observations and watch the control center respond.

In [ ]:
# Add varying loads to different stations
station_loads = [
    (entry_stations[0], 280),  # Above threshold
    (entry_stations[1], 150),  # Below threshold
    (entry_stations[2], 310),  # Above threshold
]

print("Setting up passenger loads at stations:\n")

for station, passenger_count in station_loads:
    # Clear previous queue
    queue = sim_state.get_station_queue(station.name)
    queue.waiting_passengers = []
    
    # Add new passengers
    passengers = [
        Passenger(f"{station.name}-p-{i}", station.name, route[-1].name, datetime.now())
        for i in range(passenger_count)
    ]
    queue.add_passengers(passengers)
    
    print(f"  {station.name}: {passenger_count} passengers waiting")

print()

In [ ]:
# Track dispatch requests
dispatch_log = []

# Have sensors publish observations
print("Sensors publishing observations...\n")

for sensor in sensor_agents:
    queue = sim_state.get_station_queue(sensor.station.name)
    waiting_count = sensor.count_waiting(queue)
    avg_wait_time = queue.average_wait_time_seconds
    
    # Publish observation
    sensor.publish_observation(waiting_count, avg_wait_time)
    
    print(f"Sensor at {sensor.station.name}:")
    print(f"  Waiting: {waiting_count} passengers")
    print(f"  Published to MQTT ✓")
    
    # Control center evaluates
    should_dispatch = control_center.evaluate_threshold(sensor.station.name, waiting_count)
    
    if should_dispatch:
        print(f"  ⚠️ Control center: THRESHOLD EXCEEDED")
        control_center.request_extra_train(sensor.station.name, waiting_count)
        print(f"  ➜ Dispatch request sent")
        
        dispatch_log.append({
            "station": sensor.station.name,
            "waiting_count": waiting_count,
            "timestamp": datetime.now(),
        })
    else:
        print(f"  ✓ Control center: within normal capacity")
    
    print()
    time.sleep(0.5)

## Step 7: Review Dispatch Decisions

Let's review the dispatch decisions made by the control center.

In [ ]:
print(f"Dispatch Log ({len(dispatch_log)} requests):\n")

if dispatch_log:
    for i, decision in enumerate(dispatch_log, 1):
        print(f"{i}. {decision['station']}")
        print(f"   Waiting: {decision['waiting_count']} passengers")
        print(f"   Time: {decision['timestamp'].strftime('%H:%M:%S')}")
        print()
else:
    print("  No extra trains were requested (all stations within capacity)")

## Step 8: Test Cooldown Period

The control center has a cooldown period (5 minutes) to prevent dispatching multiple trains too quickly for the same station.

In [ ]:
# Try to dispatch again immediately for same station
test_station = entry_stations[0]
queue = sim_state.get_station_queue(test_station.name)

print(f"Testing cooldown period for {test_station.name}:\n")
print(f"  Current waiting: {queue.count} passengers")
print(f"  Threshold: {threshold}")

# First request (should succeed)
result1 = control_center.evaluate_threshold(test_station.name, queue.count)
print(f"\n  First dispatch evaluation: {result1}")

if result1:
    control_center.request_extra_train(test_station.name, queue.count)
    print(f"  ✓ Request sent")

# Immediate second request (should fail due to cooldown)
result2 = control_center.evaluate_threshold(test_station.name, queue.count)
print(f"\n  Immediate second evaluation: {result2}")
print(f"  → Cooldown prevents rapid repeated dispatches")

## Cleanup

In [ ]:
# Stop control center
control_center.stop()

# Disconnect from MQTT
if connector.client and connector.client.is_connected():
    connector.disconnect()
    print("✓ Disconnected from MQTT broker")

## Exercise: Experiment with Threshold Logic

Try these experiments:

1. **Change threshold**: Modify the threshold in `config.yaml` to 200 or 300 and see how dispatch behavior changes
2. **Multiple stations over threshold**: Add passengers to all entry stations above threshold and observe dispatch requests
3. **Real-time monitoring**: Subscribe to dispatch request messages in another notebook or MQTT client

Example MQTT topic to monitor:
```
train_network/control_center/dispatch_request
```

Example for experiment 2:
```python
# Set all stations to critical load
for station in entry_stations:
    queue = sim_state.get_station_queue(station.name)
    queue.waiting_passengers = []
    
    passengers = [
        Passenger(f"{station.name}-critical-{i}", station.name, route[-1].name, datetime.now())
        for i in range(350)  # Well above threshold
    ]
    queue.add_passengers(passengers)

# Evaluate all stations
for station in entry_stations:
    queue = sim_state.get_station_queue(station.name)
    if control_center.evaluate_threshold(station.name, queue.count):
        control_center.request_extra_train(station.name, queue.count)
        print(f"✓ Dispatched for {station.name}")
```

## Next Steps

Continue to **05_train_full_simulation.ipynb** to see all agents working together in a complete, orchestrated simulation with visualization.